In [35]:
import re, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report

config

In [36]:
TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"
OUTPUT_CSV = "submission_llm.csv"
CHART_PATH = "f1_chart_llm.png"
MODEL_NAME = "microsoft/deberta-v3-small" #"roberta-base" or "roberta-base" can be an alternative too

SEED         = 42
MAX_LEN      = 512       # 512 and 256 should be checked
BATCH_SIZE   = 16
EPOCHS       = 5
LR           = 2e-5      # check other lr too
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP    = 1.0
USE_AMP      = False
CLASS_NAMES  = ["Human", "DeepSeek", "Grok", "Claude", "Gemini", "ChatGPT"]
NUM_CLASSES  = len(CLASS_NAMES)
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device  : {DEVICE}")
print(f"Model   : {MODEL_NAME}")
print(f"Max len : {MAX_LEN}  |  Batch: {BATCH_SIZE}  |  Epochs: {EPOCHS}")

Device  : cuda
Model   : microsoft/deberta-v3-small
Max len : 512  |  Batch: 16  |  Epochs: 5


Dataset

In [37]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }


class InferDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
        }


In [38]:
def compute_class_weights(labels):
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    weights = counts.sum() / (NUM_CLASSES * counts)

    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)

In [39]:
def run_infer(model, loader):
    model.eval()
    all_preds = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)

        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=attn_mask)
            preds = out.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)

    return all_preds

Training

In [40]:
def run_epoch(model, loader, criterion, optimiser=None,
              scheduler=None, scaler=None, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE) if "labels" in batch else None

        with torch.set_grad_enabled(train):

            # Forward pass
            out = model(input_ids=input_ids, attention_mask=attn_mask)

            if labels is not None:
                loss = criterion(out.logits, labels)

            # Backpropagation
            if train:
                optimiser.zero_grad()
                loss.backward()

                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

                optimiser.step()

                if scheduler is not None:
                    scheduler.step()

        if labels is not None:
            total_loss += loss.item() * len(labels)

            preds = out.logits.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    per_class = f1_score(
        all_labels,
        all_preds,
        average=None,
        labels=list(range(NUM_CLASSES)),
        zero_division=0
    )

    return avg_loss, macro_f1, per_class, all_preds, all_labels

chart

In [41]:
def plot_results(history, save_path):
    COLORS = ["#4C72B0","#55A868","#C44E52","#8172B2","#937860","#DA8BC3"]
    epochs  = range(1, len(history["train_loss"]) + 1)
    best_ep = int(np.argmax(history["val_f1"]))
    best_f1 = history["val_f1"][best_ep]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor("#0f1117")
    for ax in axes:
        ax.set_facecolor("#1a1d27")
        ax.tick_params(colors="#cccccc")
        ax.xaxis.label.set_color("#cccccc")
        ax.yaxis.label.set_color("#cccccc")
        ax.title.set_color("white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")

    fig.suptitle(
        f"Fine-tuned {MODEL_NAME}  —  AI Text Source Detector",
        fontsize=13, fontweight="bold", color="white", y=1.02,
    )

    # Loss
    ax = axes[0]
    ax.plot(epochs, history["train_loss"], "o-", label="Train",
            color="#4C72B0", lw=2, ms=5)
    ax.plot(epochs, history["val_loss"],   "s--", label="Val",
            color="#DD8452", lw=2, ms=5)
    ax.set_title("Cross-Entropy Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(facecolor="#2a2d3a", labelcolor="white")
    ax.grid(True, alpha=0.2, color="#555")

    # Macro F1
    ax = axes[1]
    ax.plot(epochs, history["train_f1"], "o-", label="Train",
            color="#4C72B0", lw=2, ms=5)
    ax.plot(epochs, history["val_f1"],   "s--", label="Val",
            color="#DD8452", lw=2, ms=5)
    ax.axhline(best_f1, color="#55A868", ls=":", lw=1.5,
               label=f"Best = {best_f1:.4f}")
    ax.set_title("Macro F1 Score"); ax.set_xlabel("Epoch"); ax.set_ylabel("Macro F1")
    ax.set_ylim(0.5, 1.02)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.legend(facecolor="#2a2d3a", labelcolor="white")
    ax.grid(True, alpha=0.2, color="#555")
    ax.annotate(
        f"  Best: {best_f1:.4f}",
        xy=(best_ep + 1, best_f1),
        xytext=(best_ep + 1.4, best_f1 - 0.07),
        arrowprops=dict(arrowstyle="->", color="#ff6b6b", lw=1.5),
        color="#ff6b6b", fontsize=9, fontweight="bold",
    )

    # Per-class bar
    ax = axes[2]
    pc = history["val_per_class"][best_ep]
    bars = ax.bar(CLASS_NAMES, pc, color=COLORS, edgecolor="#0f1117", width=0.65)
    ax.set_title(f"Per-Class F1  |  Macro = {best_f1:.4f}")
    ax.set_xlabel("Class"); ax.set_ylabel("F1 Score")
    ax.set_ylim(0, 1.18)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.grid(axis="y", alpha=0.2, color="#555")
    ax.tick_params(axis="x", colors="#cccccc", rotation=10)
    for bar, val in zip(bars, pc):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.025,
            f"{val:.3f}", ha="center", va="bottom",
            fontsize=9, fontweight="bold", color="white",
        )

    plt.tight_layout()
    plt.savefig(save_path, dpi=160, bbox_inches="tight", facecolor="#0f1117")
    plt.close()
    print(f"Chart saved → {save_path}")

In [42]:
def main():
    # Load data
    train_df   = pd.read_csv(TRAIN_PATH)
    test_df    = pd.read_csv(TEST_PATH)
    texts_all  = train_df["TEXT"].fillna("").tolist()
    labels_all = train_df["LABEL"].values
    texts_test = test_df["TEXT"].fillna("").tolist()
    test_ids = test_df["ID"].values if "ID" in test_df.columns else test_df.get("Unnamed: 0", pd.RangeIndex(len(test_df))).values

    print(f"\nTrain: {len(texts_all)}  |  Test: {len(texts_test)}")
    dist = dict(zip(*np.unique(labels_all, return_counts=True)))
    print(f"Class distribution: { {CLASS_NAMES[k]: v for k, v in dist.items()} }")

    # Tokenizer
    print(f"\nLoading tokenizer: {MODEL_NAME} ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Train / val split
    tr_texts, va_texts, tr_labels, va_labels = train_test_split(
        texts_all, labels_all,
        test_size=0.2, stratify=labels_all, random_state=SEED,
    )

    print("Tokenizing datasets...")
    tr_ds  = TextDataset(tr_texts,  tr_labels.tolist(), tokenizer)
    va_ds  = TextDataset(va_texts,  va_labels.tolist(), tokenizer)
    te_ds  = InferDataset(texts_test, tokenizer)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    te_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Model
    print(f"Loading model: {MODEL_NAME} ...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES,
    ).to(DEVICE)

    model = model.float()

    model.gradient_checkpointing_enable()

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {total_params:,}")

    # Class-weighted loss
    class_weights = compute_class_weights(labels_all)

    criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

    # Optimizer (lower LR on backbone),(higher on classifier head)
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped = [
        {"params": [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay) and "classifier" not in n],
         "lr": LR, "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay) and "classifier" not in n],
         "lr": LR, "weight_decay": 0.0},
        {"params": [p for n, p in model.named_parameters() if "classifier" in n],
         "lr": LR * 5, "weight_decay": WEIGHT_DECAY},   # classifier head at 5× LR
    ]
    optimiser = AdamW(optimizer_grouped)

    total_steps   = len(tr_loader) * EPOCHS
    warmup_steps  = int(total_steps * WARMUP_RATIO)
    scheduler     = get_cosine_schedule_with_warmup(
        optimiser, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
    )
    scaler = GradScaler() if USE_AMP else None

    # Training loop
    history = {
        "train_loss": [], "val_loss": [],
        "train_f1":   [], "val_f1":   [],
        "val_per_class": [],
    }
    best_val_f1  = 0.0
    best_weights = None

    hdr = f"{'Ep':>3} | {'TrLoss':>7} | {'TrF1':>6} | {'VaLoss':>7} | {'VaF1':>6} | Time"
    print(f"\n{hdr}\n" + "─" * len(hdr))

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_f1, _, _, _ = run_epoch(
            model, tr_loader, criterion, optimiser, scheduler, scaler, train=True)
        va_loss, va_f1, va_pc, val_preds, val_true = run_epoch(
            model, va_loader, criterion, train=False)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_f1"].append(tr_f1)
        history["val_f1"].append(va_f1)
        history["val_per_class"].append(va_pc)

        if va_f1 > best_val_f1:
            best_val_f1  = va_f1
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f"{epoch:>3} | {tr_loss:>7.4f} | {tr_f1:>6.4f} | "
              f"{va_loss:>7.4f} | {va_f1:>6.4f} | {time.time()-t0:.1f}s")

    print(f"\n✓ Best Val Macro F1: {best_val_f1:.4f}")

    # Reload best checkpoint
    model.load_state_dict(best_weights)

    # Final val report
    _, _, _, val_preds, val_true = run_epoch(model, va_loader, criterion, train=False)
    print("\n── Per-class report (best checkpoint, validation set) ──")
    print(classification_report(val_true, val_preds,
                                target_names=CLASS_NAMES, digits=4))

    # Save chart
    plot_results(history, CHART_PATH)

    # Predict on test set
    print("Generating test predictions...")
    test_preds = run_infer(model, te_loader)
    submission = pd.DataFrame({"ID": test_ids, "LABEL": test_preds})
    submission.to_csv(OUTPUT_CSV, index=False)
    print(f"✓ Submission saved → {OUTPUT_CSV}  ({len(submission)} rows)")
    print("Prediction distribution:\n", submission["LABEL"].value_counts().sort_index())


if __name__ == "__main__":
    main()



Train: 2400  |  Test: 600
Class distribution: {'Human': np.int64(1520), 'DeepSeek': np.int64(80), 'Grok': np.int64(160), 'Claude': np.int64(80), 'Gemini': np.int64(240), 'ChatGPT': np.int64(320)}

Loading tokenizer: microsoft/deberta-v3-small ...
Tokenizing datasets...
Loading model: microsoft/deberta-v3-small ...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight     

Trainable params: 141,899,526

 Ep |  TrLoss |   TrF1 |  VaLoss |   VaF1 | Time
────────────────────────────────────────────────
  1 |  1.4454 | 0.5679 |  0.8320 | 0.8099 | 184.6s
  2 |  0.7621 | 0.8665 |  0.6965 | 0.9060 | 194.5s
  3 |  0.6273 | 0.9707 |  0.6307 | 0.9408 | 197.2s
  4 |  0.5563 | 0.9941 |  0.6549 | 0.9297 | 200.9s
  5 |  0.5488 | 0.9961 |  0.6245 | 0.9577 | 201.0s

✓ Best Val Macro F1: 0.9577

── Per-class report (best checkpoint, validation set) ──
              precision    recall  f1-score   support

       Human     1.0000    1.0000    1.0000       304
    DeepSeek     0.8235    0.8750    0.8485        16
        Grok     0.9655    0.8750    0.9180        32
      Claude     1.0000    1.0000    1.0000        16
      Gemini     0.9600    1.0000    0.9796        48
     ChatGPT     1.0000    1.0000    1.0000        64

    accuracy                         0.9875       480
   macro avg     0.9582    0.9583    0.9577       480
weighted avg     0.9878    0.9875    0.98